# Wake Word Dataset Generation & Augmentation Pipeline

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/tts2ww_full_pipeline.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/tts2ww_full_pipeline.ipynb)

## Start here

**What it does.** It builds a larger practice set for a wake-word model: spoken examples of your wake phrase, each in a new voice that the OmniVoice speech generator designs from a short description, and (when you give it folders) noisy and reverberant copies.

**How long.** The quick demo that runs by default takes a few minutes on a Kaggle T4 GPU. A GPU is required: OmniVoice speaks every clip. A full set (`SMOKE = False`, 900 examples per phrase) is not timed.

**What you must set.** Nothing. Internet access must be on (Kaggle: Settings panel, Internet on) and a GPU must be selected (Kaggle: Settings panel, Accelerator, GPU. Colab: Runtime menu, Change runtime type, T4 GPU, Save). To use another phrase, change `WW_WORDS` in the Settings cell; words are joined by an underscore, for example `ok_computer`.

**What you will see at the end.** A summary table with the number of files each stage wrote. Stages that need folders you did not provide (voice cloning, noise, benchmark) print that they were skipped.

Press **Run All** (Kaggle: Run, then Run All; Colab: Runtime, then Run all).

Generates, augments and benchmarks wake-word datasets: OmniVoice speech synthesis with voice design, voiceclonnx voice-conversion augmentation, and training-time noise/reverb/pitch augmentation.

| Field | Value |
|---|---|
| Inputs | a target wake word and, optionally, a folder of donor voices for voice conversion |
| Outputs | a training dataset (audio plus `metadata.csv`, `path,label`; every row is a rendering of a wake word in `WW_WORDS` with label 1, with no per-word field, and the notebook makes no negatives) and, when `BENCH_CLEAN_DIR` is set (skipped by default), a fixed-SNR benchmark set, both under the configured output directory |
| Runtime | Quick demo (the default, `SMOKE = True`, `N_SAMPLES` clips per phrase, voice conversion skipped): a few minutes on a Kaggle T4 GPU. Full pipeline (`SMOKE = False`): not measured; it scales with `SYNTH_N` and the number of wake and reference voices, and a GPU shortens the voice-conversion stages |
| Hardware | a GPU (Kaggle T4 or better) for OmniVoice; `ONNX_RUNTIME=cuda` or `rocm` also puts the voice-conversion stages on it |
| Cost | free on Kaggle; not measured for a full run with voice conversion enabled |

CI: linted and compiled under Python 3.11 and 3.12. OmniVoice needs a GPU, so the notebook is run on Kaggle, not in CI.

**Read more:**

- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/)
- [wakeforge](https://github.com/TigreGotico/wakeforge)
- [OmniVoice](https://github.com/k2-fsa/OmniVoice)
- [voiceclonnx](https://github.com/TigreGotico/voiceclonnx)
- [Pre-wake VAD](https://openvoiceos.github.io/ovos-blogs/posts/2025-11-06-prewake-vad)

In [ ]:
SMOKE = True       # True: quick demo (a few minutes). False: the full 900 examples per phrase, much longer.
N_SAMPLES = 32


## 0 · Settings

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before** running the config cell.

**For a first run you do not need to change anything: click Run All.** The two settings people change are `WW_WORDS` (the wake phrase, words joined by an underscore) and, for a language other than English, `WW_LANG`. Everything marked empty is an optional stage that is skipped.

In [ ]:
import os

def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


# ──────────────────────────────────────────────
# GENERAL
# ──────────────────────────────────────────────
os.environ.setdefault("WW_LANG",            "en")                 # Language code (en, es, pt, …)
os.environ.setdefault("WW_WORDS",           "hey_mycroft")        # Space-separated wake words (use _ for multi-word)
os.environ.setdefault("WW_BASE_DIR",        _default_base_dir("ww"))  # Root directory for all I/O

# ──────────────────────────────────────────────
# TTS SYNTHESIS + VOICE CONVERSION  (Stage 1)
#   OmniVoice speaks every clip on the GPU. SYNTH_VOICES is a list of voice
#   descriptions separated by ";"; each clip draws one at random and OmniVoice
#   samples a fresh voice from it, so voices drawn from the same description
#   may resemble each other. The default names no accent, so it suits any
#   language OmniVoice speaks.
# ──────────────────────────────────────────────
os.environ.setdefault(
    "SYNTH_VOICES",
    "female, young adult, moderate pitch;"
    "male, young adult, low pitch;"
    "female, middle-aged, high pitch;"
    "male, middle-aged, moderate pitch;"
    "female, elderly, low pitch;"
    "male, elderly, very low pitch;"
    "female, teenager, high pitch;"
    "male, teenager, moderate pitch",
)  # OmniVoice voice descriptions, separated by ";"
os.environ.setdefault("SYNTH_N",            "900")                # Samples per wake word
os.environ.setdefault("SYNTH_SEED",         "")                   # Random seed (empty = none)
os.environ.setdefault("SYNTH_VC_REFS",      "")                   # Reference voice WAVs dir (empty = no VC)

# ──────────────────────────────────────────────
# VOICE CONVERSION ENGINE  (Stages 1, 2, 3)
#   voiceclonnx.VoiceCloner(engine=VC_ENGINE, **kwargs). One env var per
#   constructor option of the SELECTED engine, named VC_<OPTION>; an empty
#   value means "use the engine's own default". Setting an option the
#   selected engine does not accept stops the notebook, naming the engine
#   and the option. `voiceclonnx list` (CLI) prints every registered engine.
# ──────────────────────────────────────────────
os.environ.setdefault("VC_ENGINE", "chatterbox")   # voiceclonnx engine alias

os.environ.setdefault("VC_QUANTIZED",     "")   # bool  — chatterbox, cosyvoice, bicodec, facodec, focalcodec, knnvc, lscodec, openvoice, rvc, triaan
os.environ.setdefault("VC_EXAGGERATION",  "")   # float — chatterbox only
os.environ.setdefault("VC_ODE_STEPS",     "")   # int   — cosyvoice only
os.environ.setdefault("VC_CHUNK_SAMPLES", "")   # int   — bicodec only
os.environ.setdefault("VC_K",             "")   # int   — focalcodec, knnvc
os.environ.setdefault("VC_DEFAULT_MODEL", "")   # str   — rvc only
os.environ.setdefault("VC_SAMPLE_RATE",   "")   # int   — rvc only
os.environ.setdefault("VC_SPEAKER_ID",    "")   # int   — rvc only
os.environ.setdefault("VC_F0_UP_KEY",     "")   # int   — rvc only

os.environ.setdefault("ONNX_PROVIDERS", "auto")   # comma list of onnxruntime providers, or "auto"
os.environ.setdefault("ONNX_RUNTIME",   "cpu")    # cpu | cuda | rocm — which onnxruntime build the install cell fetches

# ──────────────────────────────────────────────
# VOICE CLONING AUGMENTATION  (Stage 2)
# ──────────────────────────────────────────────
os.environ.setdefault("VC_AUG_N_RANDOM",    "1")                  # Random voice refs per sample
os.environ.setdefault("VC_AUG_VOICES_PATH", "")                   # Voice reference dataset dir

# ──────────────────────────────────────────────
# VC TTS SYNTHESIS  (Stage 3)
# ──────────────────────────────────────────────
os.environ.setdefault("VC_TTS_VOICES_PATH",      "")              # Voice references dir
os.environ.setdefault("VC_TTS_EXAGG_RANGE",      "0.4 0.9 0.1")  # Exaggeration range: min max step (VC_ENGINE=chatterbox only)

# ──────────────────────────────────────────────
# TRAINING AUGMENTATION  (Stage 4)
# ──────────────────────────────────────────────
os.environ.setdefault("AUG_BG_NOISE_DIR",   "")                   # General background noise
os.environ.setdefault("AUG_MUSIC_DIR",      "")                   # Music folder
os.environ.setdefault("AUG_BG_SPEECH_DIR",  "")                   # Competing-speaker / babble
os.environ.setdefault("AUG_MIC_NOISE_DIR",  "")                   # Mic-specific silence / hiss
os.environ.setdefault("AUG_RIR_DIR",        "")                   # Room impulse responses
os.environ.setdefault("AUG_SNR_MIN",        "0")                  # dB
os.environ.setdefault("AUG_SNR_MAX",        "20")                 # dB
os.environ.setdefault("AUG_PITCH_MIN",      "-1.0")               # semitones
os.environ.setdefault("AUG_PITCH_MAX",      "1.0")
os.environ.setdefault("AUG_SPEED_MIN",      "0.95")
os.environ.setdefault("AUG_SPEED_MAX",      "1.05")
os.environ.setdefault("AUG_PROB",           "0.9")                # Per-sample augmentation probability

# ──────────────────────────────────────────────
# BENCHMARK GENERATION  (Stage 5)
# ──────────────────────────────────────────────
os.environ.setdefault("BENCH_CLEAN_DIR",    "")                   # Clean wake-word samples
os.environ.setdefault("BENCH_NOISE_DIR",    "")                   # Ambient / environmental noise
os.environ.setdefault("BENCH_MUSIC_DIR",    "")                   # Music noise
os.environ.setdefault("BENCH_SPEECH_DIR",   "")                   # Competing-speaker audio
os.environ.setdefault("BENCH_RIR_DIR",      "")                   # RIR files
os.environ.setdefault("BENCH_SNRS",         "20,15,10,5,0")       # Comma-separated SNR list (dB)

# ──────────────────────────────────────────────
# Derived paths
# ──────────────────────────────────────────────
BASE    = os.environ.get("WW_BASE_DIR", _default_base_dir("ww"))
LANG    = os.environ.get("WW_LANG", "en")
WW_LIST = os.environ.get("WW_WORDS", "hey_mycroft").split()

os.environ.setdefault("SYNTH_OUTPUT_DIR",   os.path.join(BASE, "synth_output", LANG))
os.environ.setdefault("AUG_INPUT_DIR",      os.environ["SYNTH_OUTPUT_DIR"])  # Stage 1's own output, so Stages 1 and 4 chain by default
os.environ.setdefault("VC_AUG_OUTPUT_DIR",  os.path.join(BASE, "vc_output"))
os.environ.setdefault("VC_TTS_OUTPUT_DIR",  os.path.join(BASE, "vc_synth_output"))
os.environ.setdefault("AUG_OUTPUT_DIR",     os.path.join(BASE, "augmented_data"))
os.environ.setdefault("BENCH_OUTPUT_DIR",   os.path.join(BASE, "benchmark_data"))

if SMOKE:
    # Shrink both sample-count knobs and force voice conversion off (it can
    # need a GPU for a realistic run time at full SYNTH_N).
    os.environ["SYNTH_N"]            = str(N_SAMPLES)
    os.environ["SYNTH_VC_REFS"]      = ""
    os.environ["VC_AUG_VOICES_PATH"] = ""
    os.environ["VC_TTS_VOICES_PATH"] = ""

def _env(key, default=""):
    return os.environ.get(key, default)

print("Configuration loaded.")
print(f"  Language       : {LANG}")
print(f"  Wake words     : {WW_LIST}")
print(f"  Base directory : {BASE}")

## 1 · TTS Synthesis + Voice Conversion

Speaks the wake phrase with [OmniVoice](https://github.com/k2-fsa/OmniVoice)
on the GPU, every clip in a new voice designed from a random description in
`SYNTH_VOICES`. When `SYNTH_VC_REFS` names a reference-voice directory, it
converts each rendering of the wake phrase with [voiceclonnx](https://github.com/TigreGotico/voiceclonnx).
Writes a `metadata.csv` (`path,label`) into `SYNTH_OUTPUT_DIR`, every row
labelled `1` (a rendering of the wake phrase) — this is what Stage 4 reads.

| Key env vars | |
|---|---|
| `SYNTH_VOICES` | OmniVoice voice descriptions separated by `;`, for example `female, young adult, moderate pitch`; the default names no accent and suits any language |
| `SYNTH_N` | Samples per wake word |
| `SYNTH_VC_REFS` | Reference voices dir (empty = skip voice conversion) |
| `SYNTH_SEED` | Reproducibility seed |
| `VC_ENGINE` / `VC_<OPTION>` / `ONNX_PROVIDERS` | voiceclonnx engine and its options, shared with Stages 2 and 3 |

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "OmniVoice speaks every clip and needs a GPU. On Kaggle: Settings panel, Accelerator, GPU, then Run All again. "
    "On Colab: Runtime menu, Change runtime type, choose a T4 GPU, Save, then Run all again."
)

onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

%pip install --quiet --pre "omnivoice>=0.2.1" "soundfile>=0.14.0" "voiceclonnx>=0.0.3a3" "rich>=13.7" "click>=8.1" "tqdm>=4.70.1" "ovos-utils>=0.14.1a1"

if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

In [ ]:
import csv
import inspect
import random
import sys
import time
import warnings
from os.path import join
from typing import Dict, List, Optional
from uuid import uuid4
import soundfile as sf
import torch
import torchaudio
from pathlib import Path

from tqdm import tqdm

from voiceclonnx import VoiceCloner
from voiceclonnx.engines.base import get_engine


# ═══════════════════════════════════════════════════════════════════
# Voice conversion engine — shared by Stages 1, 2 and 3
# ═══════════════════════════════════════════════════════════════════

# option name -> typed-parsing kind; the env var is VC_<OPTION UPPERCASE>
VC_OPTION_TYPES = {
    "quantized":     "bool",
    "exaggeration":  "float",
    "ode_steps":     "int",
    "chunk_samples": "int",
    "k":             "int",
    "default_model": "str",
    "sample_rate":   "int",
    "speaker_id":    "int",
    "f0_up_key":     "int",
}


def _parse_typed_env(env_name, raw, kind):
    if kind == "bool":
        low = raw.strip().lower()
        if low in ("1", "true", "yes", "on"):
            return True
        if low in ("0", "false", "no", "off"):
            return False
        raise ValueError(f"{env_name}={raw!r} is not a valid bool (use true/false)")
    if kind == "int":
        try:
            return int(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid int") from None
    if kind == "float":
        try:
            return float(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid float") from None
    return raw


def _resolve_onnx_providers(raw):
    """A comma list, or "auto": CUDA, ROCm, MIGraphX, CPU, CPU always last."""
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort

        preferred = [
            "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider",
            "CPUExecutionProvider",
        ]
        available = ort.get_available_providers()
        resolved = [p for p in preferred if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


def _engine_accepted_params(engine_alias):
    """Constructor parameter names of the engine's adapter, minus self and **cfg."""
    adapter_cls = get_engine(engine_alias).adapter_class
    try:
        sig = inspect.signature(adapter_cls.__init__)
    except (TypeError, ValueError):
        return set()
    return {
        name for name, param in sig.parameters.items()
        if name != "self" and param.kind != inspect.Parameter.VAR_KEYWORD
    }


def _engine_accepts_providers(engine_alias):
    return "providers" in _engine_accepted_params(engine_alias)


def _resolve_engine_kwargs(engine_alias, overrides=None):
    """Build VoiceCloner kwargs for `engine_alias` from every VC_<OPTION> env var
    that is set, refusing one the engine's own constructor does not accept.
    `overrides` replaces an option's env-var value when given (e.g. a swept
    exaggeration value in Stage 3)."""
    accepted = _engine_accepted_params(engine_alias)
    kwargs = {}
    for option, kind in VC_OPTION_TYPES.items():
        env_name = f"VC_{option.upper()}"
        raw = os.environ.get(env_name, "").strip()
        if overrides and option in overrides:
            value = overrides[option]
        elif raw:
            value = _parse_typed_env(env_name, raw, kind)
        else:
            continue
        if option not in accepted:
            raise RuntimeError(
                f"{env_name} is set but engine {engine_alias!r} does not accept an option "
                f"named {option!r}. Its constructor accepts: {sorted(accepted) or '(none)'}."
            )
        kwargs[option] = value
    return kwargs


def get_voice_cloner(overrides=None):
    """Build a VoiceCloner from VC_ENGINE and any VC_<OPTION> env vars set for it."""
    engine_alias = _env("VC_ENGINE", "chatterbox")
    kwargs = _resolve_engine_kwargs(engine_alias, overrides=overrides)
    if _engine_accepts_providers(engine_alias):
        kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS", "auto"))
    return VoiceCloner(engine=engine_alias, **kwargs)


# ═══════════════════════════════════════════════════════════════════
# OmniVoice — the speech generator of Stages 1 and 3
# ═══════════════════════════════════════════════════════════════════

SYNTH_SR = 16000          # every clip is written as 16 kHz mono 16-bit WAV
SYNTH_BATCH = 16          # clips per OmniVoice call
_tts = None


def _get_tts():
    """Load OmniVoice once, on the first clip that needs it."""
    global _tts
    if _tts is None:
        from omnivoice import OmniVoice
        _tts = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
    return _tts


def _parse_synth_voices(spec: str) -> List[str]:
    """Parse SYNTH_VOICES: OmniVoice voice descriptions separated by ";"."""
    voices = [v.strip() for v in spec.split(";") if v.strip()]
    if not voices:
        raise ValueError("SYNTH_VOICES is empty: give at least one voice description, e.g. 'female, young adult, moderate pitch'.")
    return voices


def check_voices(voices: List[str], lang: str) -> None:
    """Speak one short clip per description, so a description OmniVoice does not
    accept stops the run here with OmniVoice's own message (it names the
    unsupported item and suggests close matches)."""
    tts = _get_tts()
    for v in voices:
        try:
            tts.generate(text="hello", language=lang.split("-")[0], instruct=v)
        except ValueError as exc:
            raise ValueError(f"SYNTH_VOICES entry {v!r} is not a voice OmniVoice can design: {exc}") from exc


synth_errors: Dict[str, int] = {}   # every distinct synthesis error, and how often it happened


def _record_error(exc: Exception) -> None:
    """Print a synthesis error the first time its message appears, then only count it."""
    msg = f"{type(exc).__name__}: {exc}"
    if msg not in synth_errors:
        print(f"  OmniVoice error: {msg}")
    synth_errors[msg] = synth_errors.get(msg, 0) + 1


def speak(texts: List[str], out_paths: List[str], lang: str, voices: List[str]) -> List[bool]:
    """Speak each text with a new voice designed from a random description in
    `voices`, write it to the matching path, and return which clips were written.
    A batch that fails is retried clip by clip, so one bad clip costs only itself."""
    tts = _get_tts()
    lang = lang.split("-")[0]   # OmniVoice takes a bare language code, e.g. "en"
    done = []
    for start in range(0, len(texts), SYNTH_BATCH):
        chunk = list(zip(texts[start:start + SYNTH_BATCH], out_paths[start:start + SYNTH_BATCH]))
        instructs = [random.choice(voices) for _ in chunk]
        try:
            audios = tts.generate(text=[t for t, _ in chunk], language=lang, instruct=instructs)
        except Exception as exc:
            _record_error(exc)
            audios = []
            for (t, _), ins in zip(chunk, instructs):
                try:
                    audios.append(tts.generate(text=t, language=lang, instruct=ins)[0])
                except Exception as exc:
                    _record_error(exc)
                    audios.append(None)
        for (_, path), audio in zip(chunk, audios):
            if audio is None or len(audio) == 0:
                done.append(False)
                continue
            audio = torchaudio.functional.resample(torch.from_numpy(audio), tts.sampling_rate, SYNTH_SR).numpy()
            sf.write(path, audio, SYNTH_SR, subtype="PCM_16")
            done.append(True)
    return done


# ═══════════════════════════════════════════════════════════════════
# Synthesis yield check — shared by every synthesis loop below
# ═══════════════════════════════════════════════════════════════════

MAX_SYNTH_FAILURE_SHARE = 0.5  # a stage whose synthesis is broken fails loudly, not silently


def _check_synthesis_yield(success: int, fail: int, context: str) -> None:
    """A synthesis pass must produce at least one clip, and not fail silently
    on more than MAX_SYNTH_FAILURE_SHARE of its attempts. Every synthesis
    exception is already counted as `fail`; this is what turns that count
    into a result the notebook can fail on."""
    total = success + fail
    causes = f" Errors seen: {'; '.join(synth_errors)}" if synth_errors else ""
    assert total > 0, f"{context}: no synthesis attempts were made"
    assert success > 0, f"{context}: every synthesis attempt failed ({fail}/{total}).{causes}"
    failure_share = fail / total
    assert failure_share <= MAX_SYNTH_FAILURE_SHARE, (
        f"{context}: {fail}/{total} ({failure_share:.0%}) synthesis attempts failed, "
        f"more than the {MAX_SYNTH_FAILURE_SHARE:.0%} tolerance.{causes}"
    )


# ═══════════════════════════════════════════════════════════════════
# Core synthesis + voice conversion loop
# ═══════════════════════════════════════════════════════════════════

def synthesize_and_convert(
    wake_word: str,
    lang: str,
    output_dir: str,
    reference_voices_dir: Optional[str],
    n: int,
    voices: List[str],
):
    """
    Generate N samples with OmniVoice, each in a new voice designed from a
    random description in `voices` and, when `reference_voices_dir` is set,
    converted to a randomly chosen reference voice with voiceclonnx. Returns
    the (path, label) rows written; label is always 1, since every sample
    here is a positive rendering of `wake_word`.
    """
    vc_model = None
    ref_voices: List[str] = []

    if reference_voices_dir:
        ref_voices = [str(p) for p in Path(reference_voices_dir).glob("**/*.wav")]
        if not ref_voices:
            print(f"  No reference voices (.wav) found in {reference_voices_dir}")
            return []
        print(f"  Found {len(ref_voices)} reference voices. Loading voice cloner...")
        vc_model = get_voice_cloner()
        print(f"  Voice cloner ready: engine={vc_model.engine}  sample_rate={vc_model.sample_rate} Hz")
    else:
        print("  Voice conversion not enabled (SYNTH_VC_REFS empty).")

    os.makedirs(output_dir, exist_ok=True)
    rows = []
    success, fail = 0, 0
    start = time.time()

    phrase = wake_word.replace("_", " ").replace("-", " ")
    names = [str(uuid4())[10:] for _ in range(n)]
    tts_paths = [join(output_dir, f"{name}_tts.wav") for name in names]
    written = speak([phrase] * n, tts_paths, lang, voices)

    for name, tts_path, ok in tqdm(zip(names, tts_paths, written), total=n, desc="Synth + VC", unit="sample", file=sys.stdout):
        if not ok:
            fail += 1
            continue
        if vc_model is None:
            rows.append((os.path.abspath(tts_path), 1))
            success += 1
            continue
        vc_path = join(output_dir, f"{name}.wav")
        try:
            vc_model.clone_voice(tts_path, random.choice(ref_voices), vc_path)
            os.remove(tts_path)  # clean up intermediate TTS file
            rows.append((os.path.abspath(vc_path), 1))
            success += 1
        except Exception:
            fail += 1

    elapsed = time.time() - start
    print(f"  Completed {success} ok / {fail} failed in {elapsed:.1f}s")
    _check_synthesis_yield(success, fail, f"positive synthesis for {wake_word!r}")
    return rows


print("TTS + VC functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Run TTS synthesis (+ optional VC) for each wake word
# ═══════════════════════════════════════════════════════════════════

warnings.filterwarnings("ignore")

synth_voices = _parse_synth_voices(_env("SYNTH_VOICES"))
print(f"Synthesis voices ({len(synth_voices)}): " + "; ".join(synth_voices))
check_voices(synth_voices, LANG)

seed_val = _env("SYNTH_SEED")
if seed_val:
    random.seed(int(seed_val))
    torch.manual_seed(int(seed_val))
    print(f"Using random seed: {seed_val}")

synth_out   = _env("SYNTH_OUTPUT_DIR")
synth_n     = int(_env("SYNTH_N", "900"))
vc_refs     = _env("SYNTH_VC_REFS") or None

all_rows = []
for ww in WW_LIST:
    ww_out = os.path.join(synth_out, ww)
    print(f"\n{'='*50}")
    print(f"Synthesising positives: {ww}  ->  {ww_out}")
    print(f"{'='*50}")

    all_rows.extend(synthesize_and_convert(
        wake_word=ww,
        lang=LANG,
        output_dir=ww_out,
        reference_voices_dir=vc_refs,
        n=synth_n,
        voices=synth_voices,
    ))

os.makedirs(synth_out, exist_ok=True)
metadata_path = os.path.join(synth_out, "metadata.csv")
with open(metadata_path, "w", newline="", encoding="utf-8") as f:
    # paths are written relative to metadata.csv, so the folder can move
    csv.writer(f).writerows((os.path.relpath(path, synth_out), label) for path, label in all_rows)

assert all_rows, f"{metadata_path} holds no clip of the wake word"

print(f"\nTTS Synthesis complete. Wrote {len(all_rows)} wake-word clips to {metadata_path}")

## 2 · Voice Cloning Augmentation

Revoices every clip in Stage 1's output for a wake word with `VC_AUG_N_RANDOM`
random reference voices, using [voiceclonnx](https://github.com/TigreGotico/voiceclonnx).

| Key env vars | |
|---|---|
| `VC_AUG_VOICES_PATH` | Directory of reference voice WAVs |
| `VC_AUG_N_RANDOM` | Random voices per sample |
| `VC_ENGINE` / `VC_<OPTION>` | voiceclonnx engine and its options (Stage 0) |


In [ ]:
# -- Reinstall task-specific deps (in case this stage runs on its own) --
%pip install --quiet --pre "voiceclonnx>=0.0.3a3" "onnxruntime>=1.30.0"


In [ ]:
vc_aug_voices = _env("VC_AUG_VOICES_PATH")
vc_aug_out    = _env("VC_AUG_OUTPUT_DIR")
synth_out     = _env("SYNTH_OUTPUT_DIR")
n_random      = int(_env("VC_AUG_N_RANDOM", "1"))

if not vc_aug_voices:
    print("VC_AUG_VOICES_PATH not set — skipping voice cloning augmentation.")
else:
    ref_voices = [str(p) for p in Path(vc_aug_voices).glob("**/*.wav")]
    if not ref_voices:
        print(f"No reference voices (.wav) found in {vc_aug_voices} — skipping.")
    else:
        cloner = get_voice_cloner()
        print(f"Voice cloner ready: engine={cloner.engine}")

        metadata_path = os.path.join(synth_out, "metadata.csv")
        with open(metadata_path, newline="", encoding="utf-8") as f:
            synth_rows = [(row[0], int(row[1])) for row in csv.reader(f)]

        vc_aug_rows = []
        for ww in WW_LIST:
            ww_out = Path(vc_aug_out) / ww
            # Stage 1's own metadata rows, not a directory glob, so a WAV left over
            # from an earlier run under synth_out/<ww>/ is never picked up here.
            src_wavs = sorted(
                Path(synth_out) / relpath
                for relpath, label in synth_rows
                if label == 1 and Path(relpath).parts[0] == ww
            )
            if not src_wavs:
                print(f"  No positive rows for {ww} in {metadata_path} — skipping.")
                continue
            ww_out.mkdir(parents=True, exist_ok=True)

            n_done = 0
            for src_wav in src_wavs:
                for ref_file in random.sample(ref_voices, k=min(n_random, len(ref_voices))):
                    out_path = ww_out / f"{src_wav.stem}-{Path(ref_file).stem}.wav"
                    cloner.clone_voice(str(src_wav), ref_file, str(out_path))
                    # every source clip under synth_out/<ww>/ is a positive
                    # rendering of the wake word, so a voice-converted copy
                    # of it is label 1 too
                    vc_aug_rows.append((str(out_path.resolve()), 1))
                    n_done += 1
            print(f"  {ww}: {n_done} converted clips -> {ww_out}")

        if vc_aug_rows:
            with open(metadata_path, "a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerows((os.path.relpath(path, synth_out), label) for path, label in vc_aug_rows)
            print(f"  Appended {len(vc_aug_rows)} rows to {metadata_path}")

    print("\nVoice cloning augmentation complete.")

## 3 · VC TTS Synthesis

voiceclonnx converts speech to speech only, so this stage synthesises each
wake word once with OmniVoice, in a voice designed from the first description in
`SYNTH_VOICES`, then converts that
rendering to every reference voice in `VC_TTS_VOICES_PATH`. When `VC_ENGINE`
accepts an `exaggeration` option (chatterbox), it sweeps `VC_TTS_EXAGG_RANGE`
and writes one output per (reference voice, exaggeration) pair; otherwise it
writes one output per reference voice at the engine's own defaults.

| Key env vars | |
|---|---|
| `VC_TTS_VOICES_PATH` | Voice references directory |
| `VC_TTS_EXAGG_RANGE` | Exaggeration range (`min max step`); chatterbox only |
| `VC_ENGINE` / `VC_<OPTION>` | voiceclonnx engine and its options (Stage 0) |

In [ ]:
# -- Reinstall task-specific deps (in case this stage runs on its own) --
%pip install --quiet --pre "voiceclonnx>=0.0.3a3" "onnxruntime>=1.30.0"


In [ ]:
vc_tts_voices = _env("VC_TTS_VOICES_PATH")
vc_tts_out    = _env("VC_TTS_OUTPUT_DIR")
synth_out     = _env("SYNTH_OUTPUT_DIR")

if not vc_tts_voices:
    print("VC_TTS_VOICES_PATH not set — skipping VC TTS synthesis.")
else:
    ref_voices = [str(p) for p in Path(vc_tts_voices).glob("**/*.wav")]
    if not ref_voices:
        print(f"No reference voices (.wav) found in {vc_tts_voices} — skipping.")
    else:
        vc_engine = _env("VC_ENGINE", "chatterbox")
        accepted = _engine_accepted_params(vc_engine)
        donor_voice = _parse_synth_voices(_env("SYNTH_VOICES"))[0]

        if "exaggeration" in accepted:
            lo, hi, step = (float(x) for x in _env("VC_TTS_EXAGG_RANGE", "0.4 0.9 0.1").split())
            exagg_values = []
            v = lo
            while v <= hi + 1e-9:
                exagg_values.append(round(v, 4))
                v += step
        else:
            exagg_values = [None]
            print(f"  Engine {vc_engine!r} has no 'exaggeration' option — "
                  f"one pass per reference voice at its own defaults.")

        vc_tts_rows = []
        for ww in WW_LIST:
            ww_phrase = ww.replace("_", " ").replace("-", " ")
            ww_out = Path(vc_tts_out) / ww
            ww_out.mkdir(parents=True, exist_ok=True)

            base_wav = ww_out / f"_{ww}_donor.wav"
            assert speak([ww_phrase], [str(base_wav)], LANG, [donor_voice])[0], (
                f"OmniVoice could not speak {ww_phrase!r}: {'; '.join(synth_errors) or 'it returned empty audio'}"
            )

            n_done = 0
            for exagg in exagg_values:
                overrides = {"exaggeration": exagg} if exagg is not None else None
                cloner = get_voice_cloner(overrides=overrides)
                for ref_file in ref_voices:
                    ref_name = Path(ref_file).stem
                    suffix = f"_{exagg:.2f}" if exagg is not None else ""
                    out_path = ww_out / f"{ref_name}{suffix}.wav"
                    cloner.clone_voice(str(base_wav), ref_file, str(out_path))
                    # a donor-voice render of the wake word converted to
                    # another voice is still a positive rendering of ww
                    vc_tts_rows.append((str(out_path.resolve()), 1))
                    n_done += 1
            base_wav.unlink(missing_ok=True)
            print(f"  {ww}: {n_done} converted clips -> {ww_out}")

        if vc_tts_rows:
            metadata_path = os.path.join(synth_out, "metadata.csv")
            with open(metadata_path, "a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerows((os.path.relpath(path, synth_out), label) for path, label in vc_tts_rows)
            print(f"  Appended {len(vc_tts_rows)} rows to {metadata_path}")

    print("\nVC TTS synthesis complete.")

## 4 · Training Data Augmentation

Applies stochastic augmentations to preprocessed training data:
background noise mixing, music mixing, competing speech, reverb, pitch shifting, and speed perturbation.

Reads a `metadata.csv` (`path,label`) from `AUG_INPUT_DIR` and writes augmented copies + new metadata to `AUG_OUTPUT_DIR`. `AUG_INPUT_DIR` defaults to Stage 1's own `SYNTH_OUTPUT_DIR`, so this stage chains onto Stage 1 without further configuration; point it elsewhere to augment a different preprocessed dataset.

| Key env vars | |
|---|---|
| `AUG_INPUT_DIR` | Preprocessed dataset with `metadata.csv` (default: Stage 1's output) |
| `AUG_BG_NOISE_DIR` / `AUG_MUSIC_DIR` / `AUG_BG_SPEECH_DIR` / `AUG_MIC_NOISE_DIR` | Noise sources |
| `AUG_RIR_DIR` | Room impulse responses |
| `AUG_SNR_MIN` / `AUG_SNR_MAX` | SNR range (dB) |
| `AUG_PITCH_MIN/MAX`, `AUG_SPEED_MIN/MAX` | Transform ranges |
| `AUG_PROB` | Per-sample augmentation probability |


In [ ]:
%pip install --quiet --pre "librosa>=0.10.2" "soundfile>=0.14.0" "numpy>=2.0" "click>=8.1" "tqdm>=4.70.1"


In [ ]:
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════
# Audio utility functions  (shared by Stages 4 and 5)
# ═══════════════════════════════════════════════════════════════════

def load_audio_mono(path, sr=16000):
    """Load an audio file, convert to mono, and resample to target sr."""
    try:
        wav, orig_sr = sf.read(str(path))
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None
    if wav.ndim > 1:
        wav = np.mean(wav, axis=1)
    if orig_sr != sr:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return wav.astype(np.float32)


def save_wav(path, wav, sr=16000):
    """Save a numpy array as a WAV file."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), wav, sr)


def mix_audio(clean, bg, snr_db):
    """
    Mix a background track into clean audio at a given SNR (dB).
    Background is tiled/cropped to match the length of the clean audio.
    """
    clean_len = len(clean)
    if len(bg) < clean_len:
        bg = np.tile(bg, int(np.ceil(clean_len / len(bg))))
    start = random.randint(0, len(bg) - clean_len)
    bg = bg[start:start + clean_len]

    rms_clean = np.sqrt(np.mean(clean ** 2) + 1e-9)
    rms_bg    = np.sqrt(np.mean(bg ** 2) + 1e-9)
    desired   = rms_clean / (10 ** (snr_db / 20.0))
    if rms_bg > 0:
        bg = bg * (desired / rms_bg)

    mixed = clean + bg
    peak = np.max(np.abs(mixed))
    if peak > 1.0:
        mixed = mixed / peak
    return mixed.astype(np.float32)


def apply_reverb(wav, rir, attenuation=0.5):
    """Convolve with a Room Impulse Response, preserving original RMS (attenuated)."""
    out = np.convolve(wav, rir)[:len(wav)]
    rms_wav = np.sqrt(np.mean(wav ** 2) + 1e-9)
    rms_out = np.sqrt(np.mean(out ** 2) + 1e-9)
    out = out * (rms_wav / rms_out) * attenuation
    return out.astype(np.float32)


def pitch_shift(wav, sr, n_steps):
    """Pitch-shift audio by n_steps semitones."""
    return librosa.effects.pitch_shift(wav, sr=sr, n_steps=n_steps).astype(np.float32)


def speed_perturb(wav, factor):
    """Time-stretch audio by the given speed factor."""
    return librosa.effects.time_stretch(wav, rate=factor).astype(np.float32)


def collect_audio_files(base_folder):
    """Recursively collect audio files (.wav .flac .mp3 .m4a .ogg)."""
    exts = [".wav", ".flac", ".mp3", ".m4a", ".ogg"]
    if not base_folder:
        return []
    p = Path(base_folder)
    if not p.exists():
        print(f"Warning: folder not found: {base_folder}")
        return []
    files = []
    for ext in exts:
        files.extend(p.rglob(f"*{ext}"))
    return sorted(files)


print("Audio utilities ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Run training augmentation
# ═══════════════════════════════════════════════════════════════════

aug_input_dir = _env("AUG_INPUT_DIR")
aug_out_dir   = _env("AUG_OUTPUT_DIR")

if not aug_input_dir:
    print("AUG_INPUT_DIR not set — skipping training augmentation.")
else:
    input_dir = Path(aug_input_dir)
    out_dir   = Path(aug_out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    metadata_in  = input_dir / "metadata.csv"
    metadata_out = out_dir / "metadata.csv"

    if not metadata_in.exists():
        raise FileNotFoundError(f"metadata.csv not found in {input_dir}")

    # Collect augmentation source files
    bg_files      = collect_audio_files(_env("AUG_BG_NOISE_DIR"))
    music_files   = collect_audio_files(_env("AUG_MUSIC_DIR"))
    speech_files  = collect_audio_files(_env("AUG_BG_SPEECH_DIR"))
    rir_files     = collect_audio_files(_env("AUG_RIR_DIR"))
    mic_files     = collect_audio_files(_env("AUG_MIC_NOISE_DIR"))

    snr_min   = float(_env("AUG_SNR_MIN", "0"))
    snr_max   = float(_env("AUG_SNR_MAX", "20"))
    pitch_min = float(_env("AUG_PITCH_MIN", "-1.0"))
    pitch_max = float(_env("AUG_PITCH_MAX", "1.0"))
    speed_min = float(_env("AUG_SPEED_MIN", "0.95"))
    speed_max = float(_env("AUG_SPEED_MAX", "1.05"))
    prob      = float(_env("AUG_PROB", "0.9"))

    with open(metadata_in, "r", encoding="utf-8", newline="") as f:
        source_data = [row for row in csv.reader(f) if row]

    print(f"Processing {len(source_data)} samples from {metadata_in}")
    print(f"Augmentation probability: {prob*100:.0f}%")

    avail = []
    if bg_files:     avail.append(f"bg_noise ({len(bg_files)})")
    if music_files:  avail.append(f"music ({len(music_files)})")
    if speech_files: avail.append(f"speech ({len(speech_files)})")
    if rir_files:    avail.append(f"rir ({len(rir_files)})")
    if mic_files:    avail.append(f"mic ({len(mic_files)})")
    avail += ["pitch_shift", "speed_perturb"]
    print(f"Available augmentations: {', '.join(avail)}")

    augmented = []
    aug_count = 0
    n_unreadable = 0

    for idx, (path_str, label) in enumerate(tqdm(source_data, desc="Augmenting", unit="sample")):
        src_path = Path(path_str)
        if not src_path.is_absolute():
            src_path = metadata_in.parent / src_path

        try:
            wav = load_audio_mono(src_path)
        except Exception as e:
            tqdm.write(f"  Could not load {src_path}: {e}")
            n_unreadable += 1
            continue
        if wav is None:
            n_unreadable += 1
            continue

        was_augmented = False

        if random.random() < prob:
            # 1. Background noise (50% chance)
            if bg_files and random.random() < 0.5:
                bg = load_audio_mono(random.choice(bg_files))
                if bg is not None:
                    wav = mix_audio(wav, bg, random.uniform(snr_min, snr_max))
                    was_augmented = True

            # Mic-specific noise (80% chance)
            if mic_files and random.random() < 0.8:
                mic = load_audio_mono(random.choice(mic_files))
                if mic is not None:
                    wav = mix_audio(wav, mic, random.uniform(snr_min, snr_max))
                    was_augmented = True

            # 2. Music mixing (50% chance, louder SNR 0-10 dB)
            if music_files and random.random() < 0.5:
                music = load_audio_mono(random.choice(music_files))
                if music is not None:
                    wav = mix_audio(wav, music, random.uniform(0.0, 10.0))
                    was_augmented = True

            # 3. Background speech (60% chance, quieter SNR 10-25 dB)
            if speech_files and random.random() < 0.6:
                speech = load_audio_mono(random.choice(speech_files))
                if speech is not None:
                    wav = mix_audio(wav, speech, random.uniform(10.0, 25.0))
                    was_augmented = True

            # 4. Reverb (30% chance)
            if rir_files and random.random() < 0.3:
                rir = load_audio_mono(random.choice(rir_files))
                if rir is not None:
                    wav = apply_reverb(wav, rir)
                    was_augmented = True

            # 5. Pitch shift (30% chance)
            if random.random() < 0.3:
                wav = pitch_shift(wav, 16000, random.uniform(pitch_min, pitch_max))
                was_augmented = True

            # 6. Speed perturbation (30% chance)
            if random.random() < 0.3:
                wav = speed_perturb(wav, random.uniform(speed_min, speed_max))
                was_augmented = True

        if was_augmented:
            aug_count += 1

        # Peak-normalize and save
        wav = wav / (np.max(np.abs(wav)) + 1e-9)
        # The row index is part of the filename: two sources under different
        # directories can share a stem (Stage 3 names every wake word's
        # output after the reference voice alone), and a bare "stem_aug.wav"
        # would let one silently overwrite the other.
        rel_path = Path("wakes_aug" if int(label) == 1 else "negatives_aug") / f"{idx:06d}_{src_path.stem}_aug.wav"
        dst_path = out_dir / rel_path
        save_wav(dst_path, wav)
        # paths are relative to this metadata.csv, as Stage 1 writes them
        augmented.append((str(rel_path), label))

    with metadata_out.open("w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(augmented)

    if source_data and n_unreadable == len(source_data):
        raise RuntimeError(
            f"Every one of {len(source_data)} rows in {metadata_in} failed to load; "
            "nothing was augmented."
        )

    print(f"\n--- Augmentation Complete ---")
    print(f"  Generated {len(augmented)} samples ({aug_count} received transformations)")
    print(f"  Unreadable source rows skipped: {n_unreadable}")
    print(f"  Metadata: {metadata_out}")


## 5 · Benchmark Dataset Generation

Creates structured test sets at fixed SNRs and with acoustic variance (pitch shifts, speed perturbations, reverb) for evaluating false-negative rates.

| Key env vars | |
|---|---|
| `BENCH_CLEAN_DIR` | Clean wake-word samples |
| `BENCH_NOISE_DIR` / `BENCH_MUSIC_DIR` / `BENCH_SPEECH_DIR` | Noise sources |
| `BENCH_RIR_DIR` | RIR files |
| `BENCH_SNRS` | Comma-separated SNR list (e.g. `20,15,10,5,0`) |

In [ ]:
%pip install --quiet --pre "librosa>=0.10.2" "soundfile>=0.14.0" "numpy>=2.0" "click>=8.1" "tqdm>=4.70.1"


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Benchmark augmentation: deterministic test sets
# ═══════════════════════════════════════════════════════════════════

def _bench_mix(clean_files, bg_files, snr_list, output_dir, noise_type):
    """
    For each clean sample, mix with a random background file
    at every SNR in snr_list.  Creates one output per (sample, SNR).
    """
    if not bg_files:
        print(f"  Skipping {noise_type}: no background files.")
        return
    if not snr_list:
        return

    print(f"  Mixing {noise_type} at SNRs: {snr_list} dB")
    for clean_path in tqdm(clean_files, desc=f"  {noise_type}"):
        clean_wav = load_audio_mono(clean_path)
        if clean_wav is None:
            continue
        bg_path = random.choice(bg_files)
        bg_wav = load_audio_mono(bg_path)
        if bg_wav is None:
            continue

        stem = clean_path.stem
        for snr in snr_list:
            mixed = mix_audio(clean_wav, bg_wav, snr)
            out_name = f"{stem}_{noise_type}_{int(snr)}db.wav"
            save_wav(output_dir / noise_type.lower() / out_name, mixed)


def _bench_variance(clean_files, rir_files, output_dir):
    """
    Apply deterministic pitch, speed, and reverb perturbations to clean samples.
    """
    PITCH_STEPS   = [-3, -2, -1, 1, 2, 3]  # semitones
    SPEED_FACTORS = [0.9, 1.1]

    print("  Applying acoustic variance (pitch / speed / reverb)")
    for clean_path in tqdm(clean_files, desc="  Variance"):
        clean_wav = load_audio_mono(clean_path)
        if clean_wav is None:
            continue
        stem = clean_path.stem

        # Pitch shifts
        for step in PITCH_STEPS:
            out = pitch_shift(clean_wav, 16000, step)
            save_wav(output_dir / "variance" / f"{stem}_pitch_{step}.wav", out)

        # Speed perturbations
        for factor in SPEED_FACTORS:
            out = speed_perturb(clean_wav, factor)
            save_wav(output_dir / "variance" / f"{stem}_speed_{factor:.1f}.wav", out)

        # Reverb (up to 3 randomly-selected RIRs per sample)
        if rir_files:
            selected = random.sample(rir_files, k=min(3, len(rir_files)))
            for rir_path in selected:
                rir_wav = load_audio_mono(rir_path)
                if rir_wav is None:
                    continue
                out = apply_reverb(clean_wav, rir_wav)
                rir_id = rir_path.stem[:8]
                save_wav(output_dir / "variance" / f"{stem}_reverb_{rir_id}.wav", out)


print("Benchmark functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Run benchmark generation
# ═══════════════════════════════════════════════════════════════════

bench_clean_dir = _env("BENCH_CLEAN_DIR")
bench_out_dir   = Path(_env("BENCH_OUTPUT_DIR"))

if not bench_clean_dir:
    print("BENCH_CLEAN_DIR not set — skipping benchmark generation.")
else:
    clean_files  = collect_audio_files(bench_clean_dir)
    noise_files  = collect_audio_files(_env("BENCH_NOISE_DIR"))
    music_files  = collect_audio_files(_env("BENCH_MUSIC_DIR"))
    speech_files = collect_audio_files(_env("BENCH_SPEECH_DIR"))
    rir_files    = collect_audio_files(_env("BENCH_RIR_DIR"))

    snr_str  = _env("BENCH_SNRS", "20,15,10,5,0")
    snr_list = [float(s.strip()) for s in snr_str.split(",") if s.strip()]

    if not clean_files:
        print("No clean files found — aborting.")
    else:
        bench_out_dir.mkdir(parents=True, exist_ok=True)
        print(f"Found {len(clean_files)} clean files. Output -> {bench_out_dir}")

        _bench_mix(clean_files, noise_files,  snr_list, bench_out_dir, "Noise")
        _bench_mix(clean_files, music_files,  snr_list, bench_out_dir, "Music")
        _bench_mix(clean_files, speech_files, snr_list, bench_out_dir, "Speech")
        _bench_variance(clean_files, rir_files, bench_out_dir)

        print(f"\nBenchmark generation complete. Files saved under {bench_out_dir}")

## 6 · Output Summary

Quick sanity check: list all output directories and count generated files.

In [ ]:
dirs_to_check = {
    "TTS Synth output":      _env("SYNTH_OUTPUT_DIR"),
    "VC augmented output":   _env("VC_AUG_OUTPUT_DIR"),
    "VC TTS output":         _env("VC_TTS_OUTPUT_DIR"),
    "Training augmented":    _env("AUG_OUTPUT_DIR"),
    "Benchmark test sets":   _env("BENCH_OUTPUT_DIR"),
}

print("=" * 60)
print("  OUTPUT SUMMARY")
print("=" * 60)

for label, d in dirs_to_check.items():
    p = Path(d)
    if p.exists():
        wav_n = len(list(p.rglob("*.wav")))
        txt_n = len(list(p.rglob("*.txt")))
        csv_n = len(list(p.rglob("*.csv")))
        parts = []
        if wav_n: parts.append(f"{wav_n} wav")
        if txt_n: parts.append(f"{txt_n} txt")
        if csv_n: parts.append(f"{csv_n} csv")
        total = wav_n + txt_n + csv_n
        print(f"  {label:30s}  {total:>6} files  ({', '.join(parts) if parts else 'empty'})")
    else:
        print(f"  {label:30s}  (not created)")

print("=" * 60)

## 7 · Where the dataset is

The folder `SYNTH_OUTPUT_DIR` holds a `metadata.csv` of `path,label` rows beside the audio. The next cell records which wake phrase the dataset was built for and prints the folder.

### If something goes wrong

| What you see | What to do |
|---|---|
| A network error while OmniVoice downloads | Switch Internet on (Kaggle: Settings panel, Internet), then Run All again. |
| `OmniVoice speaks every clip and needs a GPU` | Select a GPU (Kaggle: Settings panel, Accelerator, GPU. Colab: Runtime menu, Change runtime type, T4 GPU, Save), then Run All again. |
| The session stopped or timed out | Press Run All again. |

In [ ]:
import json

if len(WW_LIST) == 1:
    _phrase = WW_LIST[0].replace("_", " ")
    _ds = Path(_env("SYNTH_OUTPUT_DIR"))
    (_ds / "dataset.json").write_text(json.dumps({"wake_word": _phrase}, indent=2) + "\n")
    print(f"Wake phrase recorded: {_phrase!r}")
    print(f"Dataset folder: {_ds.resolve()}")
else:
    print("This dataset holds several wake phrases; no single phrase is recorded in dataset.json.")


[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.
